# 🏷️ Notebook 01 — Innholdsbasert filtering

**Popularitetslisten ga Lea blockbustere. Kan metadata gi henne noe bedre?**

Vi prøver den enkleste personlige modellen: bruk det vi *vet* om filmene
til å finne flere som ligner på det Lea allerede liker.
Embeddings og ANN ligger som valgfri appendix til slutt.

## Metadata: nyttig eller bare pent?

Innholdsbasert filtering er som å anbefale bøker basert på omslaget —
det fungerer, men det er begrenset. Vi bygger en brukerprofil fra item-features
som sjanger, tekst eller metadata.

### Styrker

- fungerer for **nye items** uten interaksjonshistorikk
- er lett å forklare og inspisere
- gir ofte god sjangermatch tidlig

### Svakheter

- er begrenset av kvaliteten på features
- kan bli smal og forutsigbar
- ser ikke mønstre som bare blir synlige når mange brukere samspiller

## Oppsett

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from src.data import load_interactions, load_item_metadata, get_genre_matrix, GENRE_COLS
from src.split import leave_one_out_split, build_sparse_matrix
from src.metrics import recall_at_k, ndcg_at_k

interactions = load_interactions()
items = load_item_metadata()
train_df, test_df = leave_one_out_split(interactions)
n_users = interactions.user_id.max() + 1
n_items = interactions.item_id.max() + 1
train_matrix = build_sparse_matrix(train_df, n_users, n_items)
genre_matrix = get_genre_matrix(items)
user_ids = test_df['user_id'].values
test_items = test_df['item_id'].values
K = 10
LEA_ID = 451

print(f'Genre-matrise: {genre_matrix.shape}')

## 🏋️ Oppgave 1 — Innholdsbasert som første personalisering

Se på Recall-kolonnen under. Endrer metadata noe for Lea?

In [ ]:
def recommend_content_based(train_matrix, genre_matrix, user_ids, k=10):
    genre_norms = np.linalg.norm(genre_matrix, axis=1, keepdims=True)
    genre_norms = np.where(genre_norms == 0, 1.0, genre_norms)
    item_profiles = genre_matrix / genre_norms
    recommendations = np.zeros((len(user_ids), k), dtype=np.int32)

    for row_index, user_id in enumerate(user_ids):
        seen = train_matrix[user_id].indices
        if len(seen) == 0:
            recommendations[row_index] = np.arange(k)
            continue
        user_profile = item_profiles[seen].mean(axis=0)
        profile_norm = np.linalg.norm(user_profile)
        if profile_norm > 0:
            user_profile = user_profile / profile_norm
        scores = item_profiles @ user_profile
        scores[seen] = -np.inf
        recommendations[row_index] = np.argsort(-scores)[:k]
    return recommendations

item_counts = np.asarray(train_matrix.sum(axis=0)).flatten()
global_ranking = np.argsort(-item_counts)

def recommend_popular(train_matrix, user_ids, k=10):
    recommendations = np.zeros((len(user_ids), k), dtype=np.int32)
    for row_index, user_id in enumerate(user_ids):
        seen = set(train_matrix[user_id].indices)
        unseen_popular = [item_id for item_id in global_ranking if item_id not in seen][:k]
        recommendations[row_index] = unseen_popular
    return recommendations

recs_pop = recommend_popular(train_matrix, user_ids, k=K)
recs_cb = recommend_content_based(train_matrix, genre_matrix, user_ids, k=K)

print(f'Popularitet:    Recall@{K}={recall_at_k(recs_pop, test_items, K):.4f}  NDCG@{K}={ndcg_at_k(recs_pop, test_items, K):.4f}')
print(f'Innholdsbasert: Recall@{K}={recall_at_k(recs_cb, test_items, K):.4f}  NDCG@{K}={ndcg_at_k(recs_cb, test_items, K):.4f}')

In [ ]:
lea_idx = np.where(user_ids == LEA_ID)[0]
if len(lea_idx) > 0:
    for name, recs in [('Popularitet', recs_pop), ('Innholdsbasert', recs_cb)]:
        lea_recs = recs[lea_idx[0]]
        titles = items.set_index('item_id').loc[lea_recs, 'title'].values
        print(f'\n{name}:')
        for rank, title in enumerate(titles, 1):
            print(f'  {rank:>2}. {title}')

> 💬 **Diskuter**
>
> 1. Hjelper metadata Lea mer enn popularitet gjør? Hvordan ser du det?
> 2. Når er innholdsbasert filtering spesielt nyttig i praksis?
> 3. Hva er den største svakheten ved å bruke bare metadata?

### ✏️ Skriveøvelse

Skriv **én setning** til Marte som forklarer forskjellen mellom popularitet
og innholdsbasert filtering — uten å bruke ordene *vektor*, *matrise* eller *cosine*.

> *«Marte, forskjellen er at ...»*

---

> *Marte:* «Greit. Metadata hjelper, men det er fortsatt for grovt.
> Hva skjer når Lea får hjelp av tusenvis av andre brukere som ligner på henne?»

**Neste steg** → `02_collaborative_filtering.ipynb`

## Valgfri appendix — embeddings og ANN

Denne delen kan tas **etter** notebook 02 hvis dere vil koble content-based thinking
til embeddings, visualisering og retrieval i produksjonsskala.

In [ ]:
from implicit.als import AlternatingLeastSquares
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

als = AlternatingLeastSquares(factors=64, regularization=0.01, iterations=15, random_state=42, use_gpu=False)
als.fit(train_matrix, show_progress=True)
item_emb = als.item_factors
pca = PCA(n_components=2, random_state=42)
item_2d = pca.fit_transform(item_emb)
dominant_genre = genre_matrix.argmax(axis=1)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(item_2d[:, 0], item_2d[:, 1], c=dominant_genre, cmap='tab20', alpha=0.3, s=5)
ax.set_title('ALS-item-embeddings (PCA)')
plt.colorbar(scatter, ax=ax, label='Sjanger-indeks')
plt.tight_layout()
plt.show()

In [ ]:
try:
    import faiss
    emb = np.ascontiguousarray(item_emb.copy(), dtype=np.float32)
    faiss.normalize_L2(emb)
    queries = np.ascontiguousarray(als.user_factors[user_ids], dtype=np.float32)
    faiss.normalize_L2(queries)

    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    _, neighbor_ids = index.search(queries, K)
    print(f'FAISS-indeks bygget. Første bruker fikk topp-{K}: {neighbor_ids[0].tolist()}')
except ImportError:
    print('faiss-cpu er ikke installert i dette miljøet.')